# DF-THC 10x: compact groups from the Hamiltonian

**A good DF-THC table differs from a poor one less in its vectors than in how its groups sit in
orbital space: each group touches few orbitals and overlaps the others little. Building that into the
seed, from the Hamiltonian alone with no trained weights, reaches the scoring shape directly, with no
merge and no shrink.** On `dfthcsos` at (9, 143, 43) with 8-bit words that seed gives **10.06×** the
reference, checked here by the official verifier.

**Four innovations, in the order the pipeline uses them.**

1. **Compactness as the measure of a table.** Vector quality does not order tables: at the scoring shape,
   how well a single vector matches the target's top eigenspace is highest for the worst fit and lowest for
   the best. Compactness does: a trained public group spreads over about 18 orbitals, a group trained from
   eigen-leaves over about 30, and the gap survives merging and shrinking (§1, `compact`).
2. **A clustered Cholesky seed.** Pivoted Cholesky leaves are local by construction, and balanced k-means on
   their orbital profiles puts leaves that share orbitals in one group. The same L-BFGS recipe then reaches
   12.26 mHa after five rounds where the eigen seed reaches 19.09, and the same Adam schedule on 27 small
   groups reaches 11.23 mHa where eigen groups reach 17.08 (§2–3, `experiments/compact_seed.py`).
3. **Dealing rows before coded rounding.** Permuting rows inside a group leaves the Hamiltonian unchanged, so
   strong rows can be spread across the 16-address blocks that share a coded pattern. The heaviest block
   falls from 4.7× to 1.5× the mean block (§4, `experiments/deal_rows.py`).
4. **An augmented-Lagrangian refit.** It holds `error_2` and `E_gap` as constraints and spends the rest of the
   error budget on `λ_eff`, where the penalty refit stopped at a kink: 47.31 to 47.16 on the (9, 143, 46)
   table (§5, `experiments/refit_al.py`).

**End to end, from the Hamiltonian to a verified circuit.**

| Step | What it does | Result at (9, 143, 43) | Origin | Section |
|---|---|---:|---|---|
| 1. Seed | Cholesky leaves clustered by orbital, OMP vectors, least-squares weights | 651 mHa | **new** | §2 |
| 2. L-BFGS | eight rounds of 60k iterations at the shape | 12.79 mHa | public | §3 |
| 3. Adam | 1M steps, rate 1e-4 to 1e-6 | 12.39 mHa | public | §4 |
| 4. Deal | rows spread by strength across the 16-address blocks | same error | **new** | §4 |
| 5. Round | 8-bit progressive rounding with coded bit 0 | 19.55 mHa | public idea, own code | §4 |
| 6. Refine | exact gauge moves on the rounded table | `λ_eff` 49.05 | public, adapted | §5 |
| 7. Refit | two augmented-Lagrangian passes | `λ_eff` 47.12, `error_2` 19.98 mHa | **new** | §5 |
| 8. Build | 8-bit circuits, then the official verifier | 16,532 Toffolis, 428 qubits, **10.06×** | public designs, own code | §6 |

This notebook builds the seeds and the final circuits live and runs the official verifier on them. Steps 2, 3
and 5 ran on an A100, and step 7 takes far longer on Kaggle's CPU than on a laptop, so their results load from
the dataset with their logs (`REFIT_LIVE = True` reruns the refit). The final build also passed the official
metric under four random salts at full shots. A second table, at (9, 143, 44) from a k-means start selected for
low overlap, reaches 10.02× (§6).

The weights are this notebook's own. The circuits and much of the method are not: the `D(w)` penalty,
progressive rounding, coded bits, the lookup design and the loaders all come from public work, credited at
the end. §2, §4 and §5 print the source of the new parts as it runs.

What this does not show: that compactness is sufficient (at equal shape a random table is more compact
than the eigen seed and fits twice as badly, §1), or anything beyond one molecule under one gate (§7).

In [ ]:
import glob, hashlib, inspect, json, math, multiprocessing, os, re, subprocess, sys, time
from pathlib import Path

import numpy as np

try:
    import optax  # noqa: F401
except ImportError:                                  # the starter kit imports it, and not every image has it
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)

# The pipeline: the attached dataset on Kaggle, the sibling folder in the repository.
PIPELINE = next((Path(h).parent for h in [*glob.glob('/kaggle/input/**/dfthc_refit/fastfit.py', recursive=True),
                                           '../dfthc-refit/fastfit.py'] if Path(h).exists()), None)
if PIPELINE is None:
    raise FileNotFoundError('the pipeline (dfthc_refit/) was not found: attach this notebook\'s dataset')
sys.path.insert(0, str(PIPELINE))
sys.path.insert(0, str(PIPELINE / 'experiments'))
import paths                                         # finds the competition data; puts the verifier on sys.path
import build, encode, fastfit
import deal_rows, df_init, refit_al                  # experiments/: the seed, the row deal, the constrained refit
from dfthcsos import verify as V                     # the official chemistry verifier

import jax.numpy as jnp
from dfthc.train_df_thc_sga import Initialize

REFIT_LIVE = False                                   # True: rerun §5's two solves (~10 min on a fast laptop, far longer here)
BUILD_ALL = False                                    # True: build and verify all four tables in §6, not only the final one
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('build')
OUT.mkdir(exist_ok=True)
N = 76
R, B, C = 9, 143, 46                                 # the shape of the controls: 414 squares, 1,287 loaded vectors
C43 = 43                                             # the final table's copies per group: 387 squares
REFERENCE_COST = 3355599588.3155365                  # the sample submission's dfthcsos cost


def find(name: str) -> Path:
    """An input file, from the attached datasets on Kaggle or a local checkout."""
    for root in ['/kaggle/input', 'dataset', '../../data/public']:   # Kaggle, then local
        hits = sorted(glob.glob(f'{root}/**/{name}', recursive=True))
        if hits:
            if len({hashlib.sha256(Path(h).read_bytes()).hexdigest() for h in hits}) > 1:
                raise ValueError(f'{name}: different files match: {hits}')
            return Path(hits[0])
    raise FileNotFoundError(f'{name}: attach the datasets listed in the last section')


ref = Initialize(str(paths.FCIDUMP), paths.UPPER_ENERGY)
ref = {k: (jnp.asarray(v, jnp.float64) if hasattr(v, 'dtype') and jnp.issubdtype(v.dtype, jnp.floating) else v)
       for k, v in ref.items()}
with np.load(paths.SHIFTS) as z:
    frozen = {k: jnp.asarray(z[k], jnp.float64) for k in ('Hsym_lower', 'H1_I')}
prob = fastfit.problem(ref, frozen, jnp.float64)     # the target in packed pair space
T, M, ii, jj = (np.asarray(x) for x in prob)
mult = np.where(ii == jj, 1.0, 2.0)                  # pair multiplicities
d = np.sqrt(mult)                                    # M = outer(mult, mult) weighs squared entries, so rows scale by √mult
Tw = d[:, None] * T * d[None, :]                     # error_2 = ‖AᵀA − Tw‖_F with the rows of A scaled by d
lam, Q = np.linalg.eigh(Tw)
Qtop = Q[:, np.argsort(-lam)[:R * C]]                # the top 414 eigenvectors


def table(name: str) -> tuple[np.ndarray, np.ndarray]:
    """Unit vectors U (R, B, N) and weights W (R, B, C) of a continuous table."""
    with np.load(find(name)) as z:
        U, W = z['U'].astype(float), z['W'].astype(float)
    return U / np.linalg.norm(U, axis=2, keepdims=True), W


def error2(U: np.ndarray, W: np.ndarray) -> float:
    """error_2 in mHa: the Frobenius residual of the squares against the target."""
    A = np.einsum('rbc,rbp->rcp', W, U[:, :, ii] * U[:, :, jj]).reshape(-1, len(ii)) * d
    return 1e3 * float(np.linalg.norm(A.T @ A - Tw))


def compact(U: np.ndarray, W: np.ndarray) -> tuple[float, float]:
    """Effective orbitals per group (exp of the entropy of its importance-weighted orbital profile) and
    the mean cosine between the groups' profiles."""
    importance = np.sum(W ** 2, axis=2)
    p = np.array([np.sum(i[:, None] * u ** 2, axis=0) for i, u in zip(importance, U)])
    p /= p.sum(axis=1, keepdims=True)
    orbitals = np.exp(-np.sum(p * np.log(p + 1e-300), axis=1))
    ov = p @ p.T
    n = np.sqrt(np.diag(ov))
    cos = ov / n[:, None] / n[None, :]
    k = len(p)
    return float(orbitals.mean()), float((cos.sum() - k) / (k * (k - 1)))


def vector_quality(U: np.ndarray) -> float:
    """Mean share of a vector's pair projector inside the target's top 414 eigenspace."""
    u = U.reshape(-1, N)
    return float(np.mean(np.sum((u[:, ii] * u[:, jj] * d @ Qtop) ** 2, axis=1)))


def dw(W: np.ndarray) -> float:
    return 0.5 * float(np.sum(np.abs(W).sum(axis=1) ** 2 - W.sum(axis=1) ** 2))


def fits(name: str) -> list[tuple[float, float]]:
    """(error_2, D) after each full-shape refit round of a long-run log."""
    pat = re.compile(r'^\s*\d+x\d+x\d+: error_2\s+[\d.]+ ->\s+([\d.]+) mHa\s+D\s+([\d.]+)')
    return [(float(a), float(b)) for a, b in (h.groups() for h in map(pat.match, find(name).read_text().splitlines()) if h)]


def curve(name: str) -> dict[int, float]:
    """iteration -> error_2 of a local CPU fit log."""
    pat = re.compile(r'^\s*(\d+): error_2\s+([\d.]+) mHa')
    out = {}
    for h in filter(None, map(pat.match, find(name).read_text().splitlines())):
        out.setdefault(int(h.group(1)), float(h.group(2)))    # a log may hold a second run after the first
    return out


def adam_run(name: str) -> tuple[float, float, float]:
    """(error_2 at the start, best error_2, final D) of an Adam log."""
    lines = find(name).read_text().splitlines()
    last = [h for h in map(re.compile(r'^\s*\d+: error_2\s+[\d.]+ mHa\s+D\s+([\d.]+).*best\s+([\d.]+)').match, lines) if h][-1]
    return float(re.search(r'error_2\s+([\d.]+)', lines[0]).group(1)), float(last.group(2)), float(last.group(1))


def rounded(name: str) -> float:
    """error_2 at the last stage of a progressive rounding log."""
    return [float(h.group(1)) for h in map(re.compile(r'^stage.*->\s+([\d.]+) mHa').match, find(name).read_text().splitlines()) if h][-1]


def show(obj, start: str | None = None, stop: str | None = None) -> None:
    """Print the source that ran, or the part of it between two markers."""
    src = inspect.getsource(obj)
    print(src[src.index(start) if start else 0:src.index(stop) if stop else len(src)])

## 1. What differs between tables

Rows are tables, in lineage order: small groups, merged in threes, shrunk or built at the scoring shape,
and a same-shape pair at the end.

* **Vector quality:** the mean share of a loaded vector's pair projector inside the target's top 414
  eigenspace. The `R·C = 414` squares can only span that much, so this is how well a single vector is
  aimed.
* **Orbitals per group:** the exponential of the entropy of a group's importance-weighted orbital
  profile: how many orbitals the group effectively lives on.
* **Overlap:** the mean cosine between two groups' profiles.

`error_2` of the public groups is the published 10-bit table, which is tuned to sit under the 20 mHa
gate, so its continuous fit is lower than the number shown.

In [ ]:
with np.load(find('dfthc_group_v2_26_60_30.npz')) as z:
    W_public = np.array(z['W'], float)
    U_public = encode.decode(np.asarray(z['angle_words'], np.int64)[N:], encode.log_depth_tree(), 10).reshape(*W_public.shape[:2], N)
    U_public = U_public / np.linalg.norm(U_public, axis=2, keepdims=True)
rows = [('public groups', U_public, W_public), ('eigen groups, Adam', *table('eigen_27x60x30.npz')), ('clustered groups, Adam', *table('cluster_27x60x30.npz')),
        ('public, merged', *table('public_9x180x90.npz')), ('eigen, merged', *table('eigen_9x180x90.npz')),
        ('public, merged, early shrink', *table('public_9x176x84.npz')), ('eigen, merged, early shrink', *table('eigen_9x173x80.npz')),
        ('public, merged and shrunk', *table('public_9x143x46.npz')),
        ('eigen seed, direct', *table('eigen_9x143x46.npz')), ('clustered seed, direct', *table('cluster_9x143x46.npz')),
        ('eigen seed, same shape', *table('eigen_9x166x70.npz')), ('random start, same shape', *table('random_9x166x70.npz'))]
print(f"{'table':28s} {'shape':14s} {'error_2':>8s} {'D(w)':>7s} {'vector':>7s} {'orbitals':>9s} {'overlap':>8s}")
for tag, U, W in rows:
    orb, ov = compact(U, W)
    print(f'{tag:28s} {str(W.shape):14s} {error2(U, W):8.2f} {dw(W):7.2f} {vector_quality(U):7.3f} {orb:9.1f} {ov:8.3f}')

Read down the last two columns. The public groups are compact from the start, and they stay well ahead of
the eigen lineage at every stage: merged (32 orbitals against 51) and early in the shrink (about 31 against
50, at nearby shapes). The eigen lineage was not shrunk further, so there is no later pair to compare.

`error_2` is not a comparison at those two stages: the public rows are the exact merge and a 12k refit, and a
120k refit takes (9, 176, 84) from 8.42 to 6.40 mHa.

Vector quality does not order the tables. At (9, 143, 46) the eigen seed has the highest and the worst fit,
and the clustered seed the lowest and the best. Along the public lineage it moves far more than compactness
does, from about 0.8 to 0.94, while the orbitals per group stay near 30.

The last two rows are the warning: at equal shape the random table is the more compact and fits more
than twice as badly. Compactness is not sufficient on its own.

Why would compactness matter? Two groups share a Gram block `G = (U_r U_qᵀ)^∘2`, so tables whose groups
overlap in orbital space are harder to fit independently. That is an explanation, not a result: what
is measured here is the correlation in this table and the control in §3.

## 2. The seed, from the Hamiltonian alone

Four steps, with no weights from any earlier table (the helpers are `experiments/df_init.py`'s):

1. Pivoted Cholesky of the weighted target gives 414 leaves. Each is a column of the target, so it is
   local: about 7 orbitals, against 44 for an eigen-leaf.
2. Balanced k-means on the leaves' orbital profiles gives 9 groups of 46 leaves.
3. Each group picks 143 shared vectors from its leaves' eigenvectors by simultaneous orthogonal
   matching pursuit.
4. Weights by weighted least squares, leaf by leaf.

Leaves are not a fit: 414 Cholesky leaves reproduce the target to about 7 mHa, against 3.5 for the top
414 eigen-leaves. The fit that follows is free to leave them. Round-robin assignment of the same
leaves is the control for the grouping.

In [ ]:
def pivoted_cholesky(K: int) -> np.ndarray:
    """K columns of the residual of Tw, each at its largest remaining diagonal."""
    diag, L = np.diag(Tw).copy(), np.zeros((len(Tw), K))
    for k in range(K):
        q = int(np.argmax(diag))
        L[:, k] = (Tw[:, q] - L[:, :k] @ L[q, :k]) / np.sqrt(diag[q])
        diag = np.maximum(diag - L[:, k] ** 2, 0)
    return L


def build_seed(leaves: list[np.ndarray], groups: list[list[int]]) -> tuple[np.ndarray, np.ndarray]:
    """Each group's shared vectors by matching pursuit over its leaves' eigenvectors, then each leaf's weights."""
    U, W = np.zeros((R, B, N)), np.zeros((R, B, len(groups[0])))
    for r, ks in enumerate(groups):
        cand = np.concatenate([np.linalg.eigh(leaves[k])[1].T for k in ks])
        U[r] = df_init.omp_bases(cand, np.stack([leaves[k][ii, jj] for k in ks]), ii, jj, mult, B, 0.0)
        pairs = (U[r][:, ii] * U[r][:, jj]) * mult[None, :]
        for c, k in enumerate(ks):
            W[r, :, c] = np.linalg.lstsq(pairs.T, leaves[k][ii, jj] * mult, rcond=None)[0]
    return U, W


K = R * C
L = pivoted_cholesky(K)
leaves = [df_init.unpack(L[:, k] / d, ii, jj) for k in range(K)]
effective = lambda ls: np.mean([np.exp(-np.sum((p := np.sum(l ** 2, 1) / np.sum(l ** 2)) * np.log(p + 1e-300))) for l in ls])   # noqa: E731
top = np.argsort(-lam)[:K]
print(f'{K} Cholesky leaves reproduce the target to {1e3 * np.linalg.norm(L @ L.T - Tw):.2f} mHa, the top {K} eigen-leaves to '
      f'{1e3 * np.sqrt(np.sum(np.delete(lam, top) ** 2)):.2f}; one leaf lives on {effective(leaves):.1f} orbitals, an eigen-leaf on '
      f'{effective([df_init.unpack(np.sqrt(lam[k]) * Q[:, k] / d, ii, jj) for k in top]):.1f}')
profile = np.array([np.sqrt(np.sum(l ** 2, axis=1)) for l in leaves])
profile /= np.linalg.norm(profile, axis=1, keepdims=True)
weight = np.array([np.sum(l ** 2) for l in leaves])
label = df_init.balanced_kmeans(profile, R, C, weight)
groups = [list(np.flatnonzero(label == r)) for r in range(R)]
U_c, W_c = build_seed(leaves, groups)
U_r, W_r = build_seed(leaves, [list(range(r, K, R)) for r in range(R)])
for tag, U, W in [('clustered', U_c, W_c), ('round-robin', U_r, W_r)]:
    orb, ov = compact(U, W)
    print(f'{tag:12s} seed error_2 {error2(U, W):8.2f} mHa, D(w) {dw(W):6.1f}, {orb:.1f} orbitals per group, overlap {ov:.3f}')
U_s, W_s = table('seed_9x143x46.npz')
print(f'the seed of the long run: same weights to {np.abs(W_c - W_s).max():.1e}, same vectors to {np.abs(np.abs(np.sum(U_c * U_s, axis=2)) - 1).max():.1e}')
K43 = R * C43                                        # pivoted Cholesky is greedy: the first 387 leaves are the leaves of a 387-leaf run
label_43 = df_init.balanced_kmeans(profile[:K43], R, C43, weight[:K43])
U_c43, W_c43 = build_seed(leaves[:K43], [list(np.flatnonzero(label_43 == r)) for r in range(R)])
U_s43, W_s43 = table('seed_9x143x43.npz')
print(f'C = 43, seed error_2 {error2(U_c43, W_c43):.2f} mHa, the seed of the final table: same weights to {np.abs(W_c43 - W_s43).max():.1e}, '
      f'same vectors to {np.abs(np.abs(np.sum(U_c43 * U_s43, axis=2)) - 1).max():.1e}')
print()
show(df_init.balanced_kmeans)

## 3. The controls

**The short one isolates the grouping.** The same 414 leaves, the same basis selection and the same
5,000 L-BFGS iterations on a CPU, with only the assignment of leaves to groups changed. The eigen seed
is alongside as the baseline.

**The long one compares whole seeds.** 60k-iteration L-BFGS rounds at the shape, `κ` 5e-5, on an A100. The two
seeds differ in leaves, grouping, basis selection (an earlier scan of that alone moved the error by about 5%
at 500 iterations) and in the pair weighting of the truncation: the eigen seed scaled each pair by its
multiplicity where the `error_2` norm needs its square root, and §2's seed uses the square root. So this shows
that the seed is better, not which part is responsible. The eigen run stopped at five rounds when Colab
released the machine; the clustered run continued to eight.

**The third is the same idea at small groups.** Adam for 3M steps from 1e-3 to 1e-5 at (27, 60, 30), from a
clustered seed and from an eigen seed. The eigen groups then had a further warm restart. The clustered seed here
came from `df_init.py` and carries the same weighting as the eigen seed, so §2's correction is not in it;
it is also a different table from the direct one and has no verified build of its own.

In [ ]:
short = {'eigen, round-robin': curve('fit_eigen.log'), 'Cholesky, round-robin': curve('fit_chol_robin.log'),
         'Cholesky, clustered': curve('fit_chol_cluster.log')}
print(f"{'iteration':>9s}" + ''.join(f'{k:>24s}' for k in short))
for it in (1000, 2000, 3000, 4000, 5000):
    print(f'{it:9d}' + ''.join(f'{c[it]:24.2f}' for c in short.values()))
eigen, cluster = fits('t9.log')[1:], fits('t9c.log')[1:]       # [0] is the 2k warm-up
print(f"\n{'round':>5s} {'eigen':>9s} {'D(w)':>7s} {'clustered':>11s} {'D(w)':>7s}")
for k in range(len(cluster)):
    a = f'{eigen[k][0]:9.2f} {eigen[k][1]:7.2f}' if k < len(eigen) else f"{'':9s} {'':7s}"
    print(f'{k + 1:5d} {a} {cluster[k][0]:11.2f} {cluster[k][1]:7.2f}')
e0, e1, e1d = adam_run('eigen_27.log')
c0, c1, c1d = adam_run('cluster_27.log')
print(f'\n(27, 60, 30), Adam: eigen groups {e0:.2f} mHa after 3M steps, {e1:.2f} (D(w) {e1d:.2f}) after a warm restart; clustered groups {c1:.2f} (D(w) {c1d:.2f}) after 3M steps')
c43 = fits('tc43.log')[1:] + fits('tc43b.log')[1:]   # the run resumed after round two; each log opens with the 2k warm-up
print(f'\n(9, 143, 43), the same eight rounds: ' + ', '.join(f'{e:.2f}' for e, _ in c43) + f' mHa, D(w) {c43[-1][1]:.2f}')
print(f"(9, 143, 44), after the same eight rounds: the first k-means start {fits('tc44b.log')[-1][0]:.2f} mHa, the selected start {adam_run('adam_44.log')[0]:.2f}")

The grouping alone is worth about 5 mHa at 5,000 iterations. Over the long run the gap between the two
seeds widens to 7 mHa, and `D(w)`, the sign-mixing term in `Λ`, halves. The clustered table at round
five is already below the 13.51 mHa of the (9, 143, 46) table reached by merging and shrinking public
groups, with no merge, no shrink and no public weights. It also stays compact: about 20 orbitals per
group at the end of the fit (§1), up from 12 in the seed.

The same recipe at three fewer copies, the seed printed at the end of §2, stops at 12.79 mHa. That run has no
eigen control of its own, so the controls above are all at (9, 143, 46).

## 4. Adam, rounding and coded bits

Adam and progressive rounding are public methods, and `quantize.py --coded` is this pipeline's implementation
of somejoint's coded bits. Dealing the rows is new. The tables below run from (9, 143, 46), where the
controls are, to the final (9, 143, 43).

* **Adam at the shape** (1M steps, `adam_train.py`, the optimizer of Low et al.) takes the table from
  12.08 to 11.69 mHa, and 12.79 to 12.39 at C = 43, where the same pass takes the public table from 13.51 to
  13.04. Its learning-rate noise moves the sum of squared copy traces away from its
  pinned value and raises `E_gap` while the rate is high, so a table should be taken only once
  the rate has fallen; at the end of this run both are back in place.
* **Rounding** (`quantize.py`) adds in quadrature: `noise = √(final² − fit²)`.
* **Dealing the rows.** Coded low bits share one affine pattern per 16 addresses, so a
  block of strong rows pays more. The fit leaves rows nearly sorted by strength, the worst layout.
  Permuting rows inside a group leaves the Hamiltonian unchanged, so `deal_rows.py` deals them
  across the blocks by strength first.

In [ ]:
U_a, W_a = table('adam_9x143x46.npz')
fit = error2(U_a, W_a)
U_a43, W_a43 = table('adam_9x143x43.npz')
fit43 = error2(U_a43, W_a43)
print(f'Adam table: error_2 {fit:.2f} mHa, D(w) {dw(W_a):.2f}, {compact(U_a, W_a)[0]:.1f} orbitals per group')
print(f'C = 43:     error_2 {fit43:.2f} mHa, D(w) {dw(W_a43):.2f}, {compact(U_a43, W_a43)[0]:.1f} orbitals per group')
print(f"{'rounding':34s} {'fit':>7s} {'rounded':>8s} {'noise':>6s}")
for tag, f, log in [('public, κ 5e-5, plain 8 bits', 13.51, 'public_round_k5.log'), ('public, κ 2e-5, plain 8 bits', 12.81, 'public_round.log'),
                    ('clustered, 12.08, plain 8 bits', fits('t9c.log')[-1][0], 'cluster_round.log'),
                    ('clustered + Adam, plain 8 bits', fit, 'adam_round.log'), ('clustered + Adam, coded bit 0', fit, 'adam_coded_round.log'),
                    ('C = 43 + Adam, coded bit 0', fit43, 'coded_round_43.log'),
                    ('C = 44, selected start, coded bit 0', adam_run('adam_44.log')[1], 'coded_round_44.log')]:
    r = rounded(log)
    print(f'{tag:34s} {f:7.2f} {r:8.2f} {math.sqrt(r * r - f * f):6.2f}')
strength = np.sum(W_a ** 2, axis=2)
spread = lambda s: max(s[j * 16:(j + 1) * 16].sum() for j in range(-(-B // 16))) * -(-B // 16) / s.sum()     # noqa: E731
print(f'heaviest 16-row block over the mean block: {np.mean([spread(s) for s in strength]):.2f} as fit, '
      f'{np.mean([spread(s[deal_rows.deal(s, 16)]) for s in strength]):.2f} dealt')
print()
show(deal_rows.deal)

The compact table rounds no worse than the public one: its noise is about 14.3 mHa against 15.1–15.2 at both
of the public table's `κ`. Its `D(w)` is about the same as the public table's at equal `κ` (5.70 before
rounding and 6.08 after, against 6.01 and 6.17). That was not obvious in advance. A cheap proxy, weights-only
refitting after nearest rounding, predicted about 11% more noise on compact tables and was wrong in
direction: only a real progressive rounding answers this.

Coded bit 0 costs about 40 mHa² here and saves 624 Toffolis (10,154 → 9,530) in the angle circuits.

At C = 43 the fit is 0.70 mHa worse and the rounded table less than 0.1 worse, because the noise fell from 15.58
to 15.13 mHa. Using the C = 46 noise, the fit alone suggested about 19.90. So the noise is not a constant of
the method: quadrature is good for planning a shape, not for predicting its margin. The third coded table, at
C = 44, landed at 19.29 against about 19.20 from C = 43's noise. Over the three the noise stays between 15.13
and 15.58 mHa.

## 5. Spending the rest of the budget on `λ_eff`

`refit_weights.py` minimizes `log λ_eff` with one-sided quadratic penalties on `E_gap` and `error_2`. On
these tables it stops within 2 iterations on the kink of the `E_gap` penalty: `λ_eff` falls to 47.31 but
`error_2` stays at 19.49 mHa, leaving the rest of the 20 mHa unspent. `refit_al.py` holds both
constraints with an augmented Lagrangian, so the solve can keep moving along the `E_gap` floor. The
two-body words stay the same; `w`, the offsets and the one-body rows change. The one-body rows are loaded
words too, so the circuits differ, but the refit does not change the Toffoli count.

The first table below is that trade on the (9, 143, 46) table, which rounded to 19.47 mHa. About a third of the
gain over the old refit's 47.31 needs no extra error: it was the kink. The budget above that buys about 0.19 of
`λ_eff` per mHa. The other direction is far dearer: asked for 19.00, the refit stalls at 19.30 with
`λ_eff` 48.44, because at fixed words the weights cannot buy back much rounding error.

At C = 43 the first pass stops differently. Its line search fails after one step, with `error_2` at
19.56 mHa, `E_gap` still above the floor and both multipliers at zero, so neither constraint is holding it
there. A second pass from that result, after the solve's exact moves, runs normally and lowers
`λ_eff` from 47.27 to 47.12. The cell below prints both passes from their logs. `REFIT_LIVE = True` reruns them:
that took about 10 minutes on a laptop and reproduced both values, and it takes far longer on Kaggle's CPU.

In [ ]:
show(refit_al.solve, '    def lagrangian', '    z = np.concatenate')
print(f"{'error budget':>12s} {'error_2':>8s} {'λ_eff':>7s}")
for t in ('19.00', '19.49', '19.75'):
    m = json.loads(re.search(r'RESULT (\{.*\})', find(f'budget_{t}.log').read_text()).group(1))
    print(f"{t:>12s} {1e3 * m['error_2']:8.2f} {m['lambda_eff']:7.3f}")
with np.load(find('final_refit.npz')) as z:
    m = refit_al.rw.official({k: np.array(z[k]) for k in z.files}, 8)
print(f"{'19.98':>12s} {1e3 * m['error_2']:8.2f} {m['lambda_eff']:7.3f}")
with np.load(find('final_refit_43.npz')) as z:
    final = {k: np.array(z[k]) for k in z.files}
shipped = refit_al.rw.official(final, 8)
print(f"\nshipped C = 43 table: λ_eff {shipped['lambda_eff']:.3f}, error_2 {1e3 * shipped['error_2']:.2f} mHa, error_1 {1e3 * shipped['error_1']:.2f} mHa, D(w) {dw(final['W']):.2f}")
for tag, name in [('first', 'refit_43_pass1.log'), ('second', 'refit_43_pass2.log')]:
    m = json.loads(re.findall(r'^RESULT (\{.*\})', find(name).read_text(), re.M)[-1])
    print(f"{tag} pass, from its log: λ_eff {m['lambda_eff']:.3f}, error_2 {1e3 * m['error_2']:.2f} mHa, error_1 {1e3 * m['error_1']:.2f} mHa")
if REFIT_LIVE:
    with np.load(find('coded_refined_43.npz')) as z:
        data = {k: np.array(z[k]) for k in z.files}
    for k in range(2):
        t0 = time.time()
        cand, best = refit_al.solve(data, 19.98, 0.0005, 200, 3000, 3, log=lambda *x: None)
        print(f"the same solve here, pass {k + 1}: λ_eff {best['lambda_eff']:.3f}, error_2 {1e3 * best['error_2']:.2f} mHa, error_1 {1e3 * best['error_1']:.2f} mHa ({time.time() - t0:.0f} s)")
        data = {**cand, 'brot': 8}

## 6. Verified

The final (9, 143, 43) table is built here and goes through the official verifier. The other three builds went
through the same verifier in the pipeline and are recorded below; `BUILD_ALL = True` rebuilds them. They are the
plain 8-bit (9, 143, 46) table, the same table with coded bit 0, dealt rows and the constrained refit, and a
(9, 143, 44) table from a selected k-means start, carried through the same steps. The plain build used the old
refit and leaves about 1.6 mHa of the budget unspent, so it is a lower bound for its table. The coded builds
also passed the official metric at full shots under random salts: four each at C = 46 and C = 43, three at C = 44.

In [ ]:
multiprocessing.set_start_method('spawn', force=True)   # the build's workers must not fork a process that has started JAX
os.environ['OPENBLAS_NUM_THREADS'] = '1'                # one BLAS thread per worker
RECORDED = {'plain, C = 46': (17425, 428, 47.288, 5.71, 18.41, 9.515), 'coded, C = 46': (16801, 428, 47.163, 13.18, 19.98, 9.894),
            'coded, C = 44': (16622, 428, 47.074, 13.39, 19.98, 10.020)}
tables = {'plain, C = 46': 'plain_refit.npz', 'coded, C = 46': 'final_refit.npz', 'coded, C = 44': 'final_refit_44.npz', 'final, C = 43': 'final_refit_43.npz'}
results = {}
for k, (tag, name) in enumerate(tables.items()):
    if not BUILD_ALL and tag in RECORDED:
        continue
    t0 = time.time()
    sub = build.build(find(name), OUT / f'dfthcsos_{k}', 8, 'log_depth', qrom='tensor')
    r = V.verify(sub)
    results[tag] = r
    print(f"{tag}: {r['status']} in {time.time() - t0:.0f} s: {r['toffoli_count']:,.0f} Toffolis ({r['resources']['angles_toffoli']:,.0f} in the angle circuits), "
          f"{r['num_qubits']} qubits, λ_eff {r['lambda_eff']:.3f}, error_1 {1e3 * r['metrics']['error_1']:.2f} mHa, error_2 {1e3 * r['metrics']['error_2']:.2f} mHa")
    print(f"       cost {r['score']:.4g} = {REFERENCE_COST / r['score']:.3f}x the reference")
print(f"\n{'recorded':16s} {'Toffolis':>9s} {'qubits':>7s} {'λ_eff':>7s} {'error_1':>8s} {'error_2':>8s} {'score':>8s}")
for tag, (t, q, l, e1, e2, x) in RECORDED.items():
    print(f'{tag:16s} {t:9,d} {q:7d} {l:7.3f} {e1:8.2f} {e2:8.2f} {x:7.3f}x')

## 7. What this shows, and what it does not

* **Shown:** on this Hamiltonian a seed built only from the target reaches a better table at the scoring
  shape than merging and shrinking trained public groups (12.08 against 13.51 mHa, and 11.69 against 13.04 after
  the same Adam pass), and it rounds no worse. Grouping leaves by orbital is responsible for part of that (§3,
  short control). Three fewer copies per group still pass the gate, and the verifier gives 10.06×. A second
  table at C = 44 reaches 10.02× by the same steps.
* **Not shown:** that the result is independent of the k-means start. At C = 44 the first start ended L-BFGS at
  13.27 mHa and the start selected for low overlap at 12.18; the first start was not carried further, so its
  final score is unknown. At C = 43 a second, lower-overlap start was no better (14.08 against 13.97 mHa at round
  two), and the final table uses the first start.
* **Not shown:** what dealing the rows is worth. Every coded rounding here was dealt, so no run rounds the same
  table with coded bits as it was fit. The block spread in §4 is the evidence for it, not a final error.
* **Not shown:** why. Compactness correlates with fit quality and a controlled change improves it, but
  a random table shows it is not sufficient, and the long control does not separate the seed's parts. A long
  run of Cholesky leaves dealt round-robin would.
* **Not shown:** that merging is unnecessary in general. It remains exact and it gave the earlier
  tables; whether merging a compact seed improves on the direct build is not settled here.
* **One molecule, one gate:** FeMoCo76 at 20 mHa on `error_2` with the symmetry shift fixed. Nothing here
  tests another system, or the correlation-energy criterion of the DF-THC paper.

## Credits and references

This notebook and its pipeline build on public work, all Apache-2.0.

* **[ahmedberatozer, DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression):**
  the `D(w)` penalty, sensitivity-ordered progressive rounding, and long refits. Its factorization
  dataset, [ahmedberatozer/df-thc-group-compression-factorization](https://www.kaggle.com/datasets/ahmedberatozer/df-thc-group-compression-factorization),
  holds the groups measured in §1: the compactness this notebook builds in was first seen there.
  [VQCC v8 components](https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components): the fine start of the freezing schedule.
* **[somejoint](https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2):** coded low bits, an
  affine lowest bit per block of 16 addresses that the loader exploits; `quantize.py --coded` is our
  implementation.
* **[mrmeows, QCC open circuits and sources](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits):** the exact
  lookup emitter's design that `tensorqrom.py` follows, and the one-body word polish that `onebody.py`
  extends. **heliosli:** J = 5 loaders. **[lightningv08](https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom):**
  five words per ANGLE circuit.
* **[kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-complete-domain-affine-compilation), Kozykorpesh and Sumit Chongder:**
  the banked QROM and the signed Givens gauge. **kozykappa and Artem1410 (anazemcev):** the gauge
  rotation of the copy columns and the effective-norm refinement, which `refine.py` adapts. **haideptry**, via
  **[phucthaiv02](https://www.kaggle.com/code/phucthaiv02/qcc-verified-portfolio-df-thc-retraining-study):** the
  per-group orthogonal gauge; phucthaiv02 also showed refitting the weights after rounding.
* **[michaelrustad](https://www.kaggle.com/code/michaelrustad/vqcc-r10-merged-group-chemistry-12-87-lb):** the `B ≥ 129` qubit
  cliff at 8 bits and L-BFGS refits. **k256net and Dmitrii Gluzdov:** the public factorization the
  measured groups descend from.
* **The hosts:** the verifier, kickmix and the starter kit.

The four innovations in the first cell are new here. A literature and notebook search found no earlier use of
this seed; that is not proof that none exists.

* Low et al., *Fast quantum simulation of electronic structure by spectrum amplification*, arXiv:2502.15882 (DF-THC-SOS, and the Adam recipe).
* Lee et al., *Even more efficient quantum computations of chemistry through tensor hypercontraction*, arXiv:2011.03494 (THC from Cholesky input).
* *Leveraging verifiable structure in quantum circuits to find more efficient compilations with agentic AI*, arXiv:2610.23832 (the competition's benchmark paper).

**Inputs:** the competition data, the seed factorization dataset above, the hosts' starter kit
([sohier/verifiable-circuit-compilation-starter-kit](https://www.kaggle.com/datasets/sohier/verifiable-circuit-compilation-starter-kit)),
and this notebook's dataset: the pipeline (`dfthc_refit/`), the tables and the run logs.